# Feature Orthogonality & IC Audit

Fetches historical data directly from Binance REST API, computes all indicators and engineered features,
and runs a comprehensive orthogonality / information-coefficient analysis.

**No database dependency** — data comes directly from Binance Futures klines API.

## Sections
1. Setup & Imports
2. Data Fetch (BTCUSDT 1h ~6 months)
3. Indicator Computation
4. Engineered Feature Computation
5. Correlation Matrix (Pearson + Spearman)
6. Information Coefficient (IC)
7. Variance Inflation Factor (VIF)
8. Feature Importance (Random Forest)
9. Cross-Asset Comparison (XRPUSDT)
10. Summary & Recommendations

In [ ]:
# Cell 1: Setup & Imports
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add src to path for project imports
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))
sys.path.insert(0, os.path.abspath('../src'))

import math
import time
from collections import deque
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import RandomForestRegressor

# Binance SDK
from binance.um_futures import UMFutures

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11

print('Setup complete.')

In [ ]:
# Cell 2: Data Fetch — Binance REST API (paginated)

_RAW_KLINE_COLUMNS = [
    'timestamp', 'open', 'high', 'low', 'close', 'volume', 'close_time',
    'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume',
    'taker_buy_quote_asset_volume', 'ignore',
]
OHLCV_COLUMNS = ['timestamp', 'open', 'high', 'low', 'close', 'volume']


def fetch_binance_ohlcv(symbol: str, interval: str, limit: int = 4400,
                        since: int | None = None) -> pd.DataFrame:
    """Paginated fetch from Binance Futures klines endpoint (no auth needed for public data)."""
    client = UMFutures()  # public data — no key needed
    all_frames = []
    fetched = 0
    cursor = since
    max_per_call = 1500

    while fetched < limit:
        batch_limit = min(max_per_call, limit - fetched)
        params = {'limit': batch_limit}
        if cursor is not None:
            params['startTime'] = cursor

        lines = client.klines(symbol, interval, **params)
        if not lines:
            break

        df = pd.DataFrame(lines, columns=_RAW_KLINE_COLUMNS)[OHLCV_COLUMNS]
        for col in OHLCV_COLUMNS:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        all_frames.append(df)
        fetched += len(df)

        last_ts = int(df['timestamp'].iloc[-1])
        if cursor is not None and last_ts <= cursor:
            break
        cursor = last_ts + 1

        if len(lines) < batch_limit:
            break

    if not all_frames:
        return pd.DataFrame(columns=OHLCV_COLUMNS)

    result = pd.concat(all_frames, ignore_index=True)
    result = result.drop_duplicates(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    return result


# Fetch ~6 months of 1h data (6 * 30 * 24 ≈ 4320 candles)
six_months_ago_ms = int((datetime.now(timezone.utc) - timedelta(days=180)).timestamp() * 1000)

print('Fetching BTCUSDT 1h ...')
df_btc = fetch_binance_ohlcv('BTCUSDT', '1h', limit=4400, since=six_months_ago_ms)
df_btc['datetime'] = pd.to_datetime(df_btc['timestamp'], unit='ms', utc=True)
print(f'  BTCUSDT: {len(df_btc)} candles, {df_btc["datetime"].min()} → {df_btc["datetime"].max()}')

print('Fetching XRPUSDT 1h ...')
df_xrp = fetch_binance_ohlcv('XRPUSDT', '1h', limit=4400, since=six_months_ago_ms)
df_xrp['datetime'] = pd.to_datetime(df_xrp['timestamp'], unit='ms', utc=True)
print(f'  XRPUSDT: {len(df_xrp)} candles, {df_xrp["datetime"].min()} → {df_xrp["datetime"].max()}')

df_btc.head()

In [ ]:
# Cell 3: Indicator Computation
#
# Instantiate each indicator with the same params as configs/features.yaml default,
# use the batch() interface for vectorized computation, then assemble a DataFrame.

from libs.features.indicators.trend.kama import KAMA
from libs.features.indicators.trend.ema import EMA
from libs.features.indicators.momentum.rsi import RSI
from libs.features.indicators.momentum.macd import MACD
from libs.features.indicators.momentum.cci import CCI
from libs.features.indicators.momentum.adx import ADX
from libs.features.indicators.momentum.mfi import MFI
from libs.features.indicators.momentum.momentum import Momentum
from libs.features.indicators.momentum.linreg import LinReg
from libs.features.indicators.volatility.atr import ATR
from libs.features.indicators.volatility.bollinger import BollingerBands
from libs.features.indicators.volatility.keltner import KeltnerChannel
from libs.features.indicators.volume.ad_line import ADLine


def compute_all_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Compute all indicators via batch() and return a features DataFrame aligned with df index."""
    n = len(df)
    close = df['close'].tolist()
    high = df['high'].tolist()
    low = df['low'].tolist()
    volume = df['volume'].tolist()
    hlc = list(zip(high, low, close))
    hlcv = list(zip(high, low, close, volume))

    features = {}

    # --- Single-value float indicators (input: close) ---
    rsi = RSI(period=14)
    features['RSI'] = rsi.batch(close)

    kama_fast = KAMA(period=5, fast_period=2, slow_period=10)
    features['KAMA_fast'] = kama_fast.batch(close)

    kama_slow = KAMA(period=30, fast_period=2, slow_period=10)
    features['KAMA_slow'] = kama_slow.batch(close)

    ema_fast = EMA(period=12)
    features['EMA_fast'] = ema_fast.batch(close)

    ema_slow = EMA(period=26)
    features['EMA_slow'] = ema_slow.batch(close)

    momentum = Momentum(period=10)
    features['Momentum'] = momentum.batch(close)

    linreg = LinReg(period=12)
    features['LinReg'] = linreg.batch(close)

    # --- Tuple-output indicators (input: close) ---
    macd = MACD(fast_period=12, slow_period=26, signal_period=9)
    macd_raw = macd.batch(close)
    features['MACD_line'] = [x[0] if x is not None else None for x in macd_raw]
    features['MACD_signal'] = [x[1] if x is not None else None for x in macd_raw]
    features['MACD_hist'] = [x[2] if x is not None else None for x in macd_raw]

    # --- HLC indicators ---
    atr = ATR(period=14)
    features['ATR'] = atr.batch(hlc)

    cci = CCI(period=5)
    features['CCI'] = cci.batch(hlc)

    adx_ind = ADX(period=14)
    adx_raw = adx_ind.batch(hlc)
    features['ADX'] = [x['adx'] if x is not None else None for x in adx_raw]
    features['plus_DI'] = [x['plus_di'] if x is not None else None for x in adx_raw]
    features['minus_DI'] = [x['minus_di'] if x is not None else None for x in adx_raw]

    # --- Tuple-output HLC indicators ---
    bb = BollingerBands(period=20, num_std=2.0)
    bb_raw = bb.batch(close)
    features['BB_middle'] = [x[0] if x is not None else None for x in bb_raw]
    features['BB_upper'] = [x[1] if x is not None else None for x in bb_raw]
    features['BB_lower'] = [x[2] if x is not None else None for x in bb_raw]

    kc = KeltnerChannel(period=20, multiplier=1.5, atr_period=14)
    kc_raw = kc.batch(hlc)
    features['KC_middle'] = [x[0] if x is not None else None for x in kc_raw]
    features['KC_upper'] = [x[1] if x is not None else None for x in kc_raw]
    features['KC_lower'] = [x[2] if x is not None else None for x in kc_raw]

    # --- HLCV indicators ---
    ad_line = ADLine()
    features['ADLine'] = ad_line.batch(hlcv)

    mfi = MFI(period=14)
    features['MFI'] = mfi.batch(hlcv)

    feat_df = pd.DataFrame(features, index=df.index)
    return feat_df


print('Computing indicators for BTCUSDT ...')
ind_btc = compute_all_indicators(df_btc)
print(f'  Shape: {ind_btc.shape}')
print(f'  Non-null counts (first few):')
print(ind_btc.count().to_string())
ind_btc.tail(3)

In [ ]:
# Cell 4: Engineered Feature Computation
#
# Replicate the EngineeredFeatureManager.compute() pipeline bar-by-bar.
# Cross-sectional features (btc_dominance_regime, altcoin_market_momentum,
# market_cap_breadth, altcoin_beta) will produce 0.0 since we have no TV data.

from libs.features.engineered.features import (
    VolumeAdjustedMomentum,
    ATRNormalizedReturn,
    ResidualMomentum,
    SqueezeIntensity,
    RegimeScore,
    MeanReversionZ,
)

# Define the engineered features we can actually compute (non-cross-sectional)
ENG_FEATURES = [
    VolumeAdjustedMomentum(),
    ATRNormalizedReturn(),
    ResidualMomentum(),
    SqueezeIntensity(),
    RegimeScore(),
    MeanReversionZ(),
]


def compute_engineered_features(df: pd.DataFrame, ind_df: pd.DataFrame) -> pd.DataFrame:
    """Compute engineered features bar-by-bar, mimicking EngineeredFeatureManager."""
    n = len(df)
    states = {feat.name: {} for feat in ENG_FEATURES}
    records = []

    for i in range(n):
        bar_data = {
            'open': df['open'].iloc[i],
            'high': df['high'].iloc[i],
            'low': df['low'].iloc[i],
            'close': df['close'].iloc[i],
            'volume': df['volume'].iloc[i],
        }

        # Build indicator outputs dict for this bar
        features = {}
        for col in ind_df.columns:
            val = ind_df[col].iloc[i]
            if pd.notna(val):
                features[col] = val

        # Re-assemble tuple outputs that engineered features expect
        if 'BB_middle' in features and 'BB_upper' in features and 'BB_lower' in features:
            features['BollingerBands'] = (features['BB_middle'], features['BB_upper'], features['BB_lower'])
        if 'KC_middle' in features and 'KC_upper' in features and 'KC_lower' in features:
            features['KeltnerChannel'] = (features['KC_middle'], features['KC_upper'], features['KC_lower'])
        if 'ADX' in features:
            # ADX is already a float scalar in ind_df — pass as-is
            pass

        row = {}
        for feat in ENG_FEATURES:
            val = feat.compute(features, bar_data, states[feat.name], index_data=None)
            row[f'eng_{feat.name}'] = val

        records.append(row)

    return pd.DataFrame(records, index=df.index)


print('Computing engineered features for BTCUSDT ...')
eng_btc = compute_engineered_features(df_btc, ind_btc)
print(f'  Shape: {eng_btc.shape}')
print(f'  Non-null counts:')
print(eng_btc.count().to_string())

# Combine raw indicators + engineered features
all_features_btc = pd.concat([ind_btc, eng_btc], axis=1)

# Add forward returns for IC analysis
all_features_btc['fwd_ret_1'] = df_btc['close'].pct_change().shift(-1)
all_features_btc['fwd_ret_5'] = df_btc['close'].pct_change(5).shift(-5)

print(f'\nCombined feature matrix: {all_features_btc.shape}')
all_features_btc.tail(3)

In [ ]:
# Cell 5: Correlation Matrix — Pearson & Spearman
#
# Exclude cross-sectional features (always 0) and forward return columns.

CROSS_SECTIONAL_COLS = []  # none computed — they'd be constant 0
META_COLS = ['fwd_ret_1', 'fwd_ret_5']
EXCLUDE_COLS = CROSS_SECTIONAL_COLS + META_COLS

feature_cols = [c for c in all_features_btc.columns if c not in EXCLUDE_COLS]
feat_matrix = all_features_btc[feature_cols].dropna()
print(f'Analysis matrix after dropna: {feat_matrix.shape}')

# Remove constant columns (e.g., if any engineered feature is always NaN after dropna)
non_const_cols = [c for c in feat_matrix.columns if feat_matrix[c].std() > 1e-12]
feat_matrix = feat_matrix[non_const_cols]
print(f'After removing constant columns: {feat_matrix.shape}')
print(f'Feature list ({len(non_const_cols)}):\n  ' + '\n  '.join(non_const_cols))

# --- Pearson ---
pearson_corr = feat_matrix.corr(method='pearson')

fig, axes = plt.subplots(1, 2, figsize=(24, 10))

mask = np.triu(np.ones_like(pearson_corr, dtype=bool), k=1)
sns.heatmap(pearson_corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=axes[0], annot_kws={'size': 7})
axes[0].set_title('Pearson Correlation', fontsize=14)

# --- Spearman ---
spearman_corr = feat_matrix.corr(method='spearman')

sns.heatmap(spearman_corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=axes[1], annot_kws={'size': 7})
axes[1].set_title('Spearman Correlation', fontsize=14)

plt.tight_layout()
plt.show()

# Flag highly correlated pairs
print('\n=== Highly Correlated Pairs (|ρ| > 0.7) ===')
high_corr_pairs = []
for i in range(len(non_const_cols)):
    for j in range(i + 1, len(non_const_cols)):
        p_val = pearson_corr.iloc[i, j]
        s_val = spearman_corr.iloc[i, j]
        if abs(p_val) > 0.7 or abs(s_val) > 0.7:
            high_corr_pairs.append({
                'Feature A': non_const_cols[i],
                'Feature B': non_const_cols[j],
                'Pearson': round(p_val, 3),
                'Spearman': round(s_val, 3),
            })

if high_corr_pairs:
    hc_df = pd.DataFrame(high_corr_pairs).sort_values('Pearson', key=abs, ascending=False)
    print(hc_df.to_string(index=False))
else:
    print('No pairs with |ρ| > 0.7 found.')

In [ ]:
# Cell 6: Information Coefficient (IC)
#
# Rank correlation of each feature vs forward 1-bar and 5-bar returns.
# IC = Spearman(rank(feature), rank(fwd_return))  per bar, then averaged.

ic_data = all_features_btc[non_const_cols + ['fwd_ret_1', 'fwd_ret_5']].dropna()
print(f'IC analysis sample size: {len(ic_data)}')

ic_results = []
for col in non_const_cols:
    for horizon, ret_col in [('1-bar', 'fwd_ret_1'), ('5-bar', 'fwd_ret_5')]:
        rho, pval = stats.spearmanr(ic_data[col], ic_data[ret_col])
        ic_results.append({
            'Feature': col,
            'Horizon': horizon,
            'IC (Spearman)': round(rho, 5),
            'p-value': round(pval, 5),
        })

ic_df = pd.DataFrame(ic_results)

# Rolling IC to compute IC_IR = mean(IC) / std(IC) over rolling windows
# Use 60-bar rolling windows for IC stability
ROLLING_WINDOW = 60

ic_ir_results = []
for col in non_const_cols:
    for horizon, ret_col in [('1-bar', 'fwd_ret_1'), ('5-bar', 'fwd_ret_5')]:
        rolling_ics = []
        for start in range(0, len(ic_data) - ROLLING_WINDOW, ROLLING_WINDOW):
            chunk = ic_data.iloc[start:start + ROLLING_WINDOW]
            rho, _ = stats.spearmanr(chunk[col], chunk[ret_col])
            if not np.isnan(rho):
                rolling_ics.append(rho)

        if rolling_ics:
            ic_mean = np.mean(rolling_ics)
            ic_std = np.std(rolling_ics)
            ic_ir = ic_mean / ic_std if ic_std > 1e-10 else 0.0
        else:
            ic_mean = ic_std = ic_ir = 0.0

        ic_ir_results.append({
            'Feature': col,
            'Horizon': horizon,
            'IC_mean': round(ic_mean, 5),
            'IC_std': round(ic_std, 5),
            'IC_IR': round(ic_ir, 4),
        })

ic_ir_df = pd.DataFrame(ic_ir_results)

# Merge
ic_full = ic_df.merge(ic_ir_df, on=['Feature', 'Horizon'])

print('\n=== IC Summary (1-bar forward returns) ===')
ic_1 = ic_full[ic_full['Horizon'] == '1-bar'].sort_values('IC (Spearman)', key=abs, ascending=False)
print(ic_1.to_string(index=False))

print('\n=== IC Summary (5-bar forward returns) ===')
ic_5 = ic_full[ic_full['Horizon'] == '5-bar'].sort_values('IC (Spearman)', key=abs, ascending=False)
print(ic_5.to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

ic_1_sorted = ic_1.sort_values('IC (Spearman)')
colors_1 = ['#d32f2f' if v < 0 else '#388e3c' for v in ic_1_sorted['IC (Spearman)']]
axes[0].barh(ic_1_sorted['Feature'], ic_1_sorted['IC (Spearman)'], color=colors_1)
axes[0].set_title('IC — 1-bar Forward Return', fontsize=13)
axes[0].axvline(x=0, color='black', linewidth=0.5)

ic_5_sorted = ic_5.sort_values('IC (Spearman)')
colors_5 = ['#d32f2f' if v < 0 else '#388e3c' for v in ic_5_sorted['IC (Spearman)']]
axes[1].barh(ic_5_sorted['Feature'], ic_5_sorted['IC (Spearman)'], color=colors_5)
axes[1].set_title('IC — 5-bar Forward Return', fontsize=13)
axes[1].axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Variance Inflation Factor (VIF)
#
# VIF > 10 indicates severe multicollinearity.

from sklearn.linear_model import LinearRegression


def compute_vif(X: pd.DataFrame) -> pd.DataFrame:
    """Compute VIF for each column in X."""
    vif_data = []
    X_arr = X.values
    for i, col in enumerate(X.columns):
        y = X_arr[:, i]
        X_others = np.delete(X_arr, i, axis=1)
        if X_others.shape[1] == 0:
            vif_data.append({'Feature': col, 'VIF': 1.0})
            continue
        reg = LinearRegression().fit(X_others, y)
        r2 = reg.score(X_others, y)
        vif = 1.0 / (1.0 - r2) if r2 < 1.0 else float('inf')
        vif_data.append({'Feature': col, 'VIF': round(vif, 2)})
    return pd.DataFrame(vif_data).sort_values('VIF', ascending=False)


# Standardize first to avoid scale issues
feat_std = (feat_matrix - feat_matrix.mean()) / feat_matrix.std()
feat_std = feat_std.replace([np.inf, -np.inf], np.nan).dropna()

print(f'VIF computation on {feat_std.shape[0]} samples, {feat_std.shape[1]} features...')
vif_df = compute_vif(feat_std)

print('\n=== Variance Inflation Factors ===')
print(vif_df.to_string(index=False))

high_vif = vif_df[vif_df['VIF'] > 10]
if len(high_vif) > 0:
    print(f'\n⚠ {len(high_vif)} features with VIF > 10 (multicollinear):')
    print(high_vif.to_string(index=False))
else:
    print('\nAll features have VIF ≤ 10 — no severe multicollinearity detected.')

# Bar chart
fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#d32f2f' if v > 10 else '#1976d2' for v in vif_df['VIF']]
ax.barh(vif_df['Feature'], vif_df['VIF'], color=colors)
ax.axvline(x=10, color='red', linestyle='--', label='VIF = 10 threshold')
ax.set_xlabel('VIF')
ax.set_title('Variance Inflation Factor per Feature')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Feature Importance — Random Forest
#
# Simple RF regressor targeting 1-bar forward return.

rf_data = all_features_btc[non_const_cols + ['fwd_ret_1']].dropna()
X_rf = rf_data[non_const_cols]
y_rf = rf_data['fwd_ret_1']

print(f'Random Forest: {X_rf.shape[0]} samples, {X_rf.shape[1]} features')

rf = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=20,
                           random_state=42, n_jobs=-1)
rf.fit(X_rf, y_rf)

importances = pd.DataFrame({
    'Feature': non_const_cols,
    'Importance': rf.feature_importances_,
}).sort_values('Importance', ascending=False)

print('\n=== RF Feature Importances (1-bar return) ===')
print(importances.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 8))
imp_sorted = importances.sort_values('Importance')
ax.barh(imp_sorted['Feature'], imp_sorted['Importance'], color='#1976d2')
ax.set_xlabel('Importance')
ax.set_title('Random Forest Feature Importances (target: 1-bar fwd return)')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: Cross-Asset Comparison — XRPUSDT
#
# Repeat IC analysis for XRPUSDT to check feature ranking stability.

print('Computing indicators for XRPUSDT ...')
ind_xrp = compute_all_indicators(df_xrp)

print('Computing engineered features for XRPUSDT ...')
eng_xrp = compute_engineered_features(df_xrp, ind_xrp)

all_features_xrp = pd.concat([ind_xrp, eng_xrp], axis=1)
all_features_xrp['fwd_ret_1'] = df_xrp['close'].pct_change().shift(-1)
all_features_xrp['fwd_ret_5'] = df_xrp['close'].pct_change(5).shift(-5)

# Use same feature set
xrp_cols = [c for c in non_const_cols if c in all_features_xrp.columns]
ic_xrp_data = all_features_xrp[xrp_cols + ['fwd_ret_1', 'fwd_ret_5']].dropna()
print(f'XRP IC analysis sample size: {len(ic_xrp_data)}')

# Compute XRP ICs
ic_xrp_results = []
for col in xrp_cols:
    for horizon, ret_col in [('1-bar', 'fwd_ret_1'), ('5-bar', 'fwd_ret_5')]:
        rho, pval = stats.spearmanr(ic_xrp_data[col], ic_xrp_data[ret_col])
        ic_xrp_results.append({
            'Feature': col,
            'Horizon': horizon,
            'IC_XRP': round(rho, 5),
        })

ic_xrp_df = pd.DataFrame(ic_xrp_results)

# Merge BTC and XRP ICs for comparison
ic_btc_slim = ic_df[['Feature', 'Horizon', 'IC (Spearman)']].rename(columns={'IC (Spearman)': 'IC_BTC'})
ic_compare = ic_btc_slim.merge(ic_xrp_df, on=['Feature', 'Horizon'], how='inner')

print('\n=== Cross-Asset IC Comparison (1-bar) ===')
cmp_1 = ic_compare[ic_compare['Horizon'] == '1-bar'].sort_values('IC_BTC', key=abs, ascending=False)
print(cmp_1.to_string(index=False))

print('\n=== Cross-Asset IC Comparison (5-bar) ===')
cmp_5 = ic_compare[ic_compare['Horizon'] == '5-bar'].sort_values('IC_BTC', key=abs, ascending=False)
print(cmp_5.to_string(index=False))

# Scatter: BTC IC vs XRP IC
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, hz in zip(axes, ['1-bar', '5-bar']):
    subset = ic_compare[ic_compare['Horizon'] == hz]
    ax.scatter(subset['IC_BTC'], subset['IC_XRP'], s=60, alpha=0.8)
    for _, row in subset.iterrows():
        ax.annotate(row['Feature'], (row['IC_BTC'], row['IC_XRP']),
                    fontsize=7, alpha=0.7)
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
            max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'k--', alpha=0.3)
    ax.set_xlabel('IC (BTCUSDT)')
    ax.set_ylabel('IC (XRPUSDT)')
    ax.set_title(f'Cross-Asset IC Stability — {hz}')

    rho_cross, _ = stats.spearmanr(subset['IC_BTC'], subset['IC_XRP'])
    ax.text(0.05, 0.95, f'Rank corr: {rho_cross:.3f}',
            transform=ax.transAxes, fontsize=11, verticalalignment='top')

plt.tight_layout()
plt.show()

## 10. Summary & Recommendations

### What was analyzed
- **Raw indicators** (21 columns): RSI, KAMA_fast, KAMA_slow, EMA_fast, EMA_slow, Momentum, LinReg, MACD (line/signal/hist), ATR, CCI, ADX, +DI, -DI, BB (mid/upper/lower), KC (mid/upper/lower), ADLine, MFI
- **Engineered features** (6 columns): volume_adjusted_momentum, atr_normalized_return, residual_momentum, squeeze_intensity, regime_score, mean_reversion_z
- **Cross-sectional features** (excluded): btc_dominance_regime, altcoin_market_momentum, market_cap_breadth, altcoin_beta — these require TradingView index data (BTC.D, TOTAL2, TOTAL3) which is unavailable in this standalone analysis. They would produce constant 0.0.

### Key outputs
1. **Correlation heatmaps** — Pearson & Spearman identify redundant feature pairs (|ρ| > 0.7).
2. **IC analysis** — rank correlation vs 1-bar and 5-bar forward returns identifies predictive features. IC_IR (IC / IC_std) measures signal consistency.
3. **VIF** — flags multicollinear features (VIF > 10) that inflate model variance.
4. **RF importance** — non-linear importance ranking complements the linear IC/VIF checks.
5. **Cross-asset stability** — comparing BTCUSDT vs XRPUSDT IC rankings checks if feature value is structural or asset-specific.

### How to interpret
- Features with **high IC, low VIF, stable across assets** are the strongest candidates.
- Features with **high VIF + high correlation with another feature** are candidates for removal or combination.
- Features with **near-zero IC on both horizons and both assets** carry no signal and add noise.
- The engineered features should add orthogonal signal beyond raw indicators. If they are highly correlated with their constituent indicators, the engineering adds no value.